In [ ]:
import altair as alt
import pandas as pd

# 1. Instalar la librería para convertir a PNG (si ya está instalada, Colab simplemente lo saltará)
!pip install vl-convert-python

# 1. Desactivar límite de filas
alt.data_transformers.disable_max_rows()

# 2. CARGAR TU BASE DE DATOS COMPLETA (Pon el nombre de tu archivo)
df = pd.read_csv('subidas_final.csv', sep=';', decimal=',', encoding='utf-8')
df['Comuna'] = df['Comuna'].str.upper()

# 3. Filtrar solo buses
df_buses = df[df['Modo'].str.contains('BUS', case=False, na=False)]

# --- INICIO DEL AGRUPAMIENTO ---
comunas_originales = ['SANTIAGO', 'MAIPÚ', 'LAS CONDES', 'PUENTE ALTO']
df_orig = df_buses[df_buses['Comuna'].isin(comunas_originales)]
df_agrupado_orig = df_orig.groupby(['Media_hora', 'Comuna'])['Subidas_Promedio'].sum().reset_index()

comunas_nuevas = ['INDEPENDENCIA', 'LO ESPEJO', 'SAN RAMÓN', 'SAN RAMON']
df_nuevas = df_buses[df_buses['Comuna'].isin(comunas_nuevas)]
df_agrupado_nuevas = df_nuevas.groupby(['Media_hora'])['Subidas_Promedio'].sum().reset_index()
df_agrupado_nuevas['Comuna'] = 'IND + ESP + S.RAMÓN'

df_final = pd.concat([df_agrupado_orig, df_agrupado_nuevas], ignore_index=True)
# --- FIN DEL AGRUPAMIENTO ---

# 🚨 LA SOLUCIÓN AL DESORDEN DE HORARIOS 🚨
# Convertimos el texto a formato de Tiempo (Datetime)
df_final['Media_hora'] = pd.to_datetime(df_final['Media_hora'], format='%H:%M:%S')


# 4. Crear el "sensor" interactivo (escáner)
hover = alt.selection_point(
    fields=['Media_hora'],
    nearest=True,
    on='mouseover',
    empty='none'
)

# Colores para cada línea
escala_colores = alt.Scale(
    domain=['SANTIAGO', 'MAIPÚ', 'LAS CONDES', 'PUENTE ALTO', 'IND + ESP + S.RAMÓN'],
    range=['#007BFF', '#ff7f0e', '#2ca02c', '#d62728', '#9b59b6']
)

# 5. CAPA 1: Líneas base (Nota el Media_hora:T para que lo lea como TIEMPO)
lineas = alt.Chart(df_final).mark_line(strokeWidth=3).encode(
    x=alt.X('Media_hora:T',
            title='Horario (24 Horas)',
            axis=alt.Axis(format='%H:%M', labelAngle=-45, labelFont='Arial', titleFontWeight='bold')),
    y=alt.Y('Subidas_Promedio:Q',
            title='Volumen de Pasajeros',
            axis=alt.Axis(format=',.0f', titleFontWeight='bold')),
    color=alt.Color('Comuna:N', scale=escala_colores, legend=alt.Legend(title="Comunas", symbolStrokeWidth=4))
)

# 6. CAPA 2: Puntos al pasar el cursor interactivo
puntos = lineas.mark_circle(size=70).encode(
    opacity=alt.condition(hover, alt.value(1), alt.value(0)),
    tooltip=[
        alt.Tooltip('Comuna:N', title='Sector'),
        alt.Tooltip('Media_hora:T', title='Hora Exacta', format='%H:%M'),
        alt.Tooltip('Subidas_Promedio:Q', title='Total Pasajeros', format=',.0f')
    ]
).add_params(hover)

# 7. CAPA 3: Línea vertical de guía
regla = alt.Chart(df_final).mark_rule(color='gray', strokeDash=[3, 3]).encode(
    x='Media_hora:T',
).transform_filter(hover)

# 10. Unir todo y dar formato
grafico_final = (lineas + puntos + regla).properties(
    width=800,
    height=450
).configure_title(
    font='Arial Narrow',
    fontSize=20,
    anchor='start',
    color='#FFCC00'
).configure_view(
    stroke=None
)

# 9. GUARDAR COMO JSON
grafico_final.save('grafico_contraste_final.json')
grafico_final.save('grafico_contraste_final.png') # Para tu informe escrito
grafico_final.save('grafico_contraste_final.html')
print("✓ Gráfico corregido y guardado como grafico_contraste_final")